# Lab 07: Prepare raw data for analysis

`trade_raw_2019_2023.csv` has been assembled from several sources and is messy. You will profile it, fix names, numbers, units and dates, deal with duplicates and missing values, combine it with reference and real World Bank data, validate it and save it, keeping a log of every change.

For each step:
1. Read the explanation. The **Example** shows the pattern to follow; look back at the matching slide or demo notebook if you need more.
2. Replace each `...` or `# TODO` with your code and run the cell (**Shift+Enter**).
3. Run the **check** cell below it: it prints `OK` when you are right, or shows an `AssertionError` if not.

Steps build on each other, so work in order. After a restart use *Run > Run All Above Selected Cell*.

---
# Part A: Profile the raw data

## A1. Set up
Loads the messy file, the country reference list and **real** World Bank data. `log` will hold one line for each cleaning step you make.

*Run this cell; no changes needed.*

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("../../data")
OUT = Path("output")
OUT.mkdir(exist_ok=True)

raw = pd.read_csv(DATA / "trade_raw_2019_2023.csv")
countries = pd.read_excel(DATA / "countries.xlsx")
wb = pd.read_csv(DATA / "wb_indicators_2015_2023.csv")
log = []
print(raw.shape)
raw.head(10)

## A2. Column types
Look at the type of `Value`. Why is it not a number?

*Run this cell; no changes needed.*

In [ ]:
raw.dtypes

## A3. Reporter spellings
`value_counts()` shows every distinct value and how often it appears. Store the counts for the `Reporter` column in `reporter_counts`.

In [ ]:
reporter_counts = ...   # TODO
reporter_counts

In [ ]:
# check
assert len(reporter_counts) > 14
print("A3 OK: more spellings than countries")

## A4. Exact duplicates
`raw.duplicated().sum()` counts rows that are exact copies of an earlier row. Store it in `n_dups`.

In [ ]:
n_dups = ...   # TODO
print(n_dups)

In [ ]:
# check
assert n_dups == 10
print("A4 OK")

**Question:** List every quality problem you can see so far (look at `Reporter`, `Period`, `Value` and `Unit`).

*Your answer:* (double-click to edit)

---
# Part B: Standardise names

## B1. Lower-case column names
Work on a copy so `raw` stays untouched. `df.columns.str.lower()` gives lower-case names.

In [ ]:
df = raw.copy()
# TODO
print(list(df.columns))

In [ ]:
# check
assert list(df.columns) == ["reporter", "partner", "period", "flow", "value", "unit"]
print("B1 OK")

## B2. Strip spaces
`.str.strip()` works on a whole column at once. Strip the `reporter` column.

In [ ]:
# TODO
df["reporter"].value_counts().head()

In [ ]:
# check
assert not df["reporter"].str.startswith(" ").any() and not df["reporter"].str.endswith(" ").any()
print("B2 OK")

## B3. Which names do not match?
Find reporter names that are **not** in `countries["country_name"]`. Use `set(a) - set(b)`.

In [ ]:
unmatched = ...   # TODO
print(sorted(unmatched))

In [ ]:
# check
assert "USA" in unmatched and "Vietnam" in unmatched
print("B3 OK")

## B4. Map the variants
Complete `NAME_MAP` so every name in B3 maps to its spelling in `countries.xlsx`, then apply it with `.replace(NAME_MAP)`.

In [ ]:
NAME_MAP = {
    "USA": "United States",
    "UK": "United Kingdom",
    # TODO: the other variants from B3
}
# TODO: apply the map to df["reporter"]
log.append("Mapped reporter name variants to countries.xlsx spellings")
print(df["reporter"].nunique(), "reporters")

In [ ]:
# check
assert set(df["reporter"]) <= set(countries["country_name"]) and df["reporter"].nunique() == 14
print("B4 OK")

---
# Part C: Numbers and units

## C1. Remove thousands separators
Make `text`: the value column as text (`.astype(str)`) with commas removed (`.str.replace(",", "", regex=False)`).

In [ ]:
text = ...   # TODO
text.head(10)

In [ ]:
# check
assert not text.str.contains(",").any()
print("C1 OK")

## C2. Convert to numbers
`pd.to_numeric(text, errors="coerce")` converts to numbers and turns anything unconvertible (`n/a`, blanks) into `NaN` (missing). Store it back in `df["value"]` and log how many are missing.

In [ ]:
df["value"] = ...   # TODO
missing = df["value"].isna().sum()
log.append(f"Converted value to numbers; {missing} blank or n/a values are now missing")
print(missing, "missing")

In [ ]:
# check
assert df["value"].dtype == float
print("C2 OK")

## C3. Convert thousands to millions
`thousands` marks the rows in USD thousands. Use `df.loc[thousands, "value"]` to divide those values by 1000, and set their unit to `"USD millions"`.

**Example**
```python
df.loc[mask, "col"] = new_value
```

In [ ]:
thousands = df["unit"] == "USD thousands"
print(thousands.sum(), "rows in thousands")
# TODO: divide their values by 1000
# TODO: set their unit
log.append(f"Converted {thousands.sum()} rows from USD thousands to USD millions")

In [ ]:
# check
assert (df["unit"] == "USD millions").all()
print("C3 OK")

## C4. Negative exports
Exports cannot be negative; these are sign errors. Count them into `n_neg`, then replace them with their absolute value (`.abs()`), in the same way as C3.

In [ ]:
negative = df["value"] < 0
n_neg = ...   # TODO
# TODO: make them positive
log.append(f"Made {n_neg} negative values positive (sign errors, flagged to the source)")
print(n_neg)

In [ ]:
# check
assert n_neg == 3 and not (df["value"] < 0).any()
print("C4 OK")

---
# Part D: Dates

## D1. Four date formats
The `period` column mixes `2021-12-31`, `31/12/2021`, `Dec 2021` and `2021`.

*Run this cell; no changes needed.*

In [ ]:
df["period"].sample(10, random_state=0).tolist()

## D2. A function that tries each format
Complete `parse_year(p)`: loop over `FORMATS`; **try** `pd.to_datetime(p, format=f)` and return its `.year`; if a `ValueError` occurs, `continue` to the next format. Return `None` if nothing matched.

In [ ]:
FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%b %Y", "%Y"]

def parse_year(p):
    for f in FORMATS:
        try:
            ...   # TODO
        except ValueError:
            continue
    return None

print(parse_year("Dec 2021"), parse_year("31/12/2020"))

In [ ]:
# check
assert parse_year("2021-12-31") == 2021 and parse_year("31/12/2020") == 2020
assert parse_year("Dec 2019") == 2019 and parse_year("2023") == 2023
print("D2 OK")

## D3. Apply it to every row
`.map(function)` calls a function on every value in a column. Create `df["year"]`.

In [ ]:
df["year"] = ...   # TODO
log.append("Parsed four period formats into a year column")
df[["period", "year"]].head()

In [ ]:
# check
assert df["year"].notna().all() and set(df["year"]) == {2019, 2020, 2021, 2022, 2023}
print("D3 OK")

---
# Part E: Duplicates and missing values

## E1. Remove exact duplicates
`drop_duplicates()` removes rows that are identical in every column. Log how many were removed.

In [ ]:
before = len(df)
df = ...   # TODO
log.append(f"Removed {before - len(df)} exact duplicate rows")
print(before, "->", len(df))

In [ ]:
# check
assert not df.duplicated().any()
print("E1 OK")

## E2. Duplicate keys
A **key** identifies a record: reporter, partner and year. Show all rows that share a key with `df[df.duplicated(key, keep=False)]`. Store them in `key_dups`.

In [ ]:
key = ["reporter", "partner", "year"]
key_dups = ...   # TODO
key_dups.sort_values(key)

In [ ]:
# check
assert len(key_dups) > 0
print("E2 OK")

**Question:** These rows were not exact duplicates in the raw file. Why do they only show up now?

*Your answer:* (double-click to edit)

## E3. Keep one row per key
Sort so rows **with** a value come first (`sort_values("value", na_position="last")`), then `drop_duplicates(key, keep="first")`.

In [ ]:
df = ...   # TODO
log.append(f"Resolved {len(key_dups)} rows sharing a key, keeping one per key")
print(len(df))

In [ ]:
# check
assert not df.duplicated(key).any() and len(df) == 420
print("E3 OK")

## E4. Flag missing values
Official statistics rarely invent numbers. Add a True/False column `value_missing` instead of deleting rows.

In [ ]:
# TODO
log.append(f"Flagged {df['value_missing'].sum()} rows with missing values")
print(df["value_missing"].sum())

In [ ]:
# check
assert df["value_missing"].sum() == df["value"].isna().sum()
print("E4 OK")

---
# Part F: Combine datasets

## F1. Add ISO3 codes and income groups
Left-merge `countries` onto `df`, matching `reporter` to `country_name`. `validate="many_to_one"` stops with an error if a country appears twice in the reference list.

In [ ]:
df = df.merge(
    countries[["iso3", "country_name", "income_group"]],
    # TODO: left_on, right_on, how, validate
)
df.head()

In [ ]:
# check
assert df["iso3"].notna().all()
print("F1 OK")

## F2. Add real GDP data
Keep the `partner == "World"` rows, merge `wb[["iso3", "year", "gdp_usd"]]` on `iso3` and `year`, then calculate `exports_pct_gdp_calc = value * 1e6 / gdp_usd * 100` (value is in USD millions).

In [ ]:
world = ...   # TODO
world = ...   # TODO: merge
# TODO: calculate
world[["reporter", "year", "exports_pct_gdp_calc"]].head()

In [ ]:
# check
assert len(world) == 70 and "exports_pct_gdp_calc" in world.columns
print("F2 OK (the trade values are illustrative; the method is what matters)")

---
# Part G: Validate and save

## G1. A validation function
`validate(df)` returns a list of problems; an empty list means ready. The first check is written; add three more: missing `iso3`, duplicate keys (`iso3`, `partner`, `year`), and any unit that is not `"USD millions"`.

In [ ]:
def validate(df):
    problems = []
    if (df["value"] < 0).any():
        problems.append("negative values")
    # TODO: three more checks
    return problems

print(validate(df) or "All checks passed")

In [ ]:
# check
bad = df.copy(); bad.loc[bad.index[0], "unit"] = "USD thousands"
assert validate(df) == [] and "mixed units" in validate(bad)
print("G1 OK")

## G2. Save the data and the log
Save `df` to `output/trade_clean.csv` (no index) and write the `log` lines to `output/cleaning_log.txt`, one per line (`"\n".join(log)`).

In [ ]:
# TODO: save the CSV
# TODO: write the log
print("\n".join(log))

In [ ]:
# check
assert (OUT / "trade_clean.csv").exists() and (OUT / "cleaning_log.txt").exists()
print("G2 OK")